# P06 SaaS を操作する ― 書き込みを安全に

> **この教材に登場する会社・人物・製品・数値・文書はすべて架空のものです。実在の企業・団体・個人とは関係ありません。**

発注 (inventory) と経費の申請 (expense) の書き込みの道具を、確認・二重登録の防止・本人だけ・金額の承認の 4 つの守りで動かします。確認 1 往復の素の線と、Tasks の有無も確かめます。

- 上から順に実行します。0 のセルは準備です (Colab でも手元の Jupyter でも同じ)。
- API キーが無ければ、講師の実行記録を再生します (費用 0)。記録も無ければ mock で最後まで流れます (mock の数字は性能の証拠になりません)。
- 🔴 データはすべて架空の会社・文書です。

## 0. 準備

In [ ]:
# 0-1. 実行場所を見分けて、キットのフォルダに移る (Colab のときだけ取得とインストールをする)
import os, sys, subprocess
REPO_URL = "https://github.com/gucchi123/mcp-skills-colab.git"
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB:
    KIT = "/content/mcp-skills-kit"
    if not os.path.exists(KIT):
        subprocess.run(["git", "clone", "-q", REPO_URL, KIT], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{KIT}/requirements.txt"], check=True)
else:  # 手元: notebooks/ から開いた前提。キットの根 (common/ がある所) を探す
    KIT = os.path.abspath(".." if os.path.exists("../common/host.py") else ".")
os.chdir(KIT)
sys.path.insert(0, KIT)
print("実行場所:", "Colab" if IN_COLAB else "手元の Jupyter", "/ キットの根:", os.path.basename(KIT))

In [ ]:
# 0-2. Claude の呼び先を決める (キーは表示しない) と、ラボのモジュールを読めるようにする
from common.nb import choose_provider, safe
from common.kit import use_labs, utf8_console
PROVIDER = choose_provider(IN_COLAB)
use_labs(); utf8_console()

## 1. モデルに見せる引数の形
Python の関数には `confirm` がありますが、モデルに見せる inputSchema には出ません。人に聞いて埋める引数だからです。

In [ ]:
from run_saas_writes import amain as writes
with safe():
    r = await writes(["--only", "schema"])

## 2. 確認の文と、承認・断る
登録の前にサーバが出す確認の文を見て、承認・断る・閉じた・承認だが ok=false の 4 通りで答えます。

In [ ]:
with safe():
    r = await writes(["--only", "confirm"])

## 3. 同じ request_id の 2 回目と、本人だけ
再試行で同じ依頼が 2 回届いても二重に登録しません。経費は本人の名前でしか申請できません。

In [ ]:
with safe():
    r = await writes(["--only", "idem,self"])

## 4. 購買の依頼 10 件を正解と照合
10 万円を超える発注は「承認待ち」で登録します。O-08 は依頼文で承認を飛ばすよう頼んでいますが、決めるのはサーバです。

In [ ]:
with safe():
    r = await writes(["--only", "gold", "--all"])

## 5. 確認 1 往復の素の線
SDK の Client とサーバの間に、行を書き写す中継をはさみます。`input_required` が返り、`inputResponses` を付けて送り直す流れがそのまま見えます。

In [ ]:
with safe():
    r = await writes(["--only", "wire"])

## 6. Tasks (長い処理) は使えるか
mcp 2.3.0 の型・クラスを調べ、サーバに `tasks/get` を送って確かめます。

In [ ]:
with safe():
    r = await writes(["--only", "tasks"])

## 7. ホストで発注係を動かす (mock)
🔴 モデルは mock (筋書きどおりの偽物) です。承認役が断る → 承認する → 同じ依頼をもう一度、の 3 回で、確認がどう出るかを見ます。

In [ ]:
with safe():
    r = await writes(["--only", "host"])